In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from cmcrameri import cm

import sys
sys.path.append("../src")

from vuln_curve_fitting import *

In [ ]:
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica"],
    "font.size": 11,
    "axes.linewidth": 1,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "xtick.major.width": 1,
    "ytick.major.width": 1,
})

## Define normal yield
Defined as
- mean detrended yield for rice/maize/sugarcane/cassava of the 30 year (1991-2020);  
- mean yield for rubber due to its short study period (2016-2020).

In [ ]:
rice_normal_yield = {
    "ID": 4.23,
    "KH": 1.26,
    "MM": 2.75,
    "PH": 2.65,
    "TH": 2.03,
    "VN": 3.09
}

In [ ]:
maize_normal_yield = {
    "ID": 1.96,
    "KH": 0.73,
    "MM": 1.17,
    "PH": 1.24,
    "TH": 2.74,
    "VN": 1.48
}

In [ ]:
rubber_normal_yield = {
    "ID": 0.9310,
    "KH": 1.1455,
    "MM": 0.7845,
    "PH": 1.7968,
    "TH": 1.4637,
    "VN": 1.6685
}

In [ ]:
sugarcane_normal_yield = {
    "ID": 80.2,
    "KH": 27.0,
    "MM": 41.7,
    "PH": 68.8,
    "TH": 45.1,
    "VN": 41.4
}

In [ ]:
cassava_normal_yield = {
    "ID": 11,
    "KH": 3.1,
    "MM": 8.7,
    "PH": 8.8,
    "TH": 12,
    "VN": 6.5
}

In [ ]:
# plot uniform colorbar
cmap = cm.batlow
norm = mpl.colors.LogNorm(vmin=100, vmax=220000)
'''
# add transparency to colorbar
colors = cmap(np.linspace(0, 1, 256))
colors[:, -1] = 0.4      # constant transparency

transparent_cmap = mpl.colors.ListedColormap(colors)
'''
# Scalar mappable for colorbar
sm = mpl.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])

# Figure (colorbar only)
fig = plt.figure(figsize=(3, 0.6))

# [left, bottom, width, height]
cax = fig.add_axes([0.1, 0.4, 0.8, 0.2])

cbar = fig.colorbar(
    sm,
    cax=cax,
    orientation="horizontal",
)

cbar.set_label("Grid HA (ha)")

plt.show()
#plt.savefig("../../fig/colorbar_HA.svg", bbox_inches="tight", dpi=300)

## Fit vuln curve
Due to varying representative power in response to spei values, for the function binned_statistics, use integer "bins" for rice/maize;
use bin edges [-3, -2, -1.5, -1, 0] for other crops.

May need to abandon the 1st point with very negative spei samples (less reliable due to small sample size).

In [ ]:
country_code = "TH"

grid_df = pd.read_csv(
    f"../../Results/3.3_VulnCurveSamples/"
    f"vuln_curve_samples_maize_{country_code.lower()}_260720.csv"
)

vuln_curve_plot_path = f"../../fig/vulcurve_maize_{country_code.lower()}.svg"

In [ ]:
# calculate yield loss ratio
grid_ls_ratio_df = calc_ls_ratio(grid_df, maize_normal_yield[country_code])

negative_spei_df = grid_ls_ratio_df[grid_ls_ratio_df["spei"]<0]

result = binned_statistics(
    negative_spei_df,
    bins=7, #[-3, -2, -1.5, -1, 0]
    alpha=0.05
)
result

In [ ]:
popt, perr, rmse, x_fit, y_fit, y_upper, y_lower = fit_logistic_ranged(
    binned_x = result["spei_bin"].apply(lambda b: b.mid).to_numpy()[1:],
    binned_y = result["weighted_mean_lr"][1:],
    p0=None,
    prepend_origin=True,
)

print(f"RMSE: {rmse:.4f}")
print(f"Fitted upper, growth_rate, midpoint: {np.round(popt, decimals=3)}")
print(f"Parameter uncertainties: {np.round(perr, decimals=3)}")

In [ ]:
x = negative_spei_df["spei"]
y = negative_spei_df["ls_ratio"]
ha = negative_spei_df["harvest_area_ha"]

binned_x = result["spei_bin"].apply(lambda b: b.mid).to_numpy()
binned_y = result["weighted_mean_lr"]
binned_y_err = result["ci_high"]-binned_y

cmap = cm.batlow
norm = mpl.colors.LogNorm(vmin=100, vmax=220000)

fig, ax = plt.subplots(figsize=(2.5, 2.5))

# Background scatter
ax.scatter(
    x, y,
    c=ha,
    s=8,
    cmap=cmap,
    norm=norm,
    alpha=0.25,
    edgecolor="none",
    zorder=1
)

# Binned data with error bars
ax.errorbar(
    binned_x,
    binned_y,
    yerr=binned_y_err,
    fmt="o",
    color="#333333",
    markeredgecolor="white",
    ecolor="#333333",
    elinewidth=1.2,
    capsize=3,
    markersize=6,
    markeredgewidth=0.8,
    zorder=3
)

ax.errorbar(
    binned_x[0],
    binned_y[0],
    yerr=binned_y_err[0],
    fmt="o",
    color="#D55E00",
    ecolor="#D55E00",
    elinewidth=1.2,
    capsize=3,
    markersize=4,
    zorder=3
)


# Confidence interval
ax.fill_between(
    x_fit,
    np.minimum(y_lower, y_fit), # avoid misleading non-S form curve
    np.maximum(y_upper, y_fit), # avoid misleading non-S form curve
    color="#0072B2",
    alpha=0.18,
    linewidth=0,
    zorder=2
)

# Fitted line
ax.plot(
    x_fit,
    y_fit,
    color="#0072B2",
    linewidth=2.5,
    zorder=4
)

ax.set_ylim([0, 1])
ax.set_xlim([-3, 0])
ax.set_xticks([-3,-2,-1,0])

ax.set_xlabel("<SPEI>", fontsize=14)
ax.set_ylabel("Yield loss ratio", fontsize=14)

ax.tick_params(axis="both", labelsize=14)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.tick_params(direction="out", length=3, width=0.8, colors="black")

ax.spines["left"].set_linewidth(0.8)
ax.spines["bottom"].set_linewidth(0.8)

plt.tight_layout()
#plt.show()
plt.savefig(vuln_curve_plot_path, dpi = 300, bbox_inches="tight")